In [9]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import pandas as pd
import json
from cassandra.query import BatchStatement

In [10]:
#Day la de ket noi voi AstraDB khi chay tren local
cloud_config= {
  'secure_connect_bundle': '/home/tee/doanbigdata/secure-connect-doanbigdata.zip'
}
with open("/home/tee/Downloads/doanbigdata-token.json") as f:
    secrets = json.load(f)

CLIENT_ID = secrets["clientId"]
CLIENT_SECRET = secrets["secret"]
#

auth_provider = PlainTextAuthProvider(CLIENT_ID, CLIENT_SECRET)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()
session.set_keyspace('final')

In [ ]:
df = pd.read_csv("/home/tee/doanbigdata/Air_quality_final.csv")

prepared_stmt = session.prepare("""
    INSERT INTO air_quality (date, time, c6h6_gt, co_gt, nmhc_gt, no2_gt, nox_gt, pt08_s1_co, pt08_s2_nmhc, pt08_s3_nox, pt08_s4_no2, pt08_s5_o3, rh, t, ah)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")
batch = BatchStatement()

for i, row in df.iterrows():
    batch.add(prepared_stmt, (row['Date'], row['Time'], row['C6H6(GT)'], row['CO(GT)'], row['NMHC(GT)'], row['NO2(GT)'], 
                              row['NOx(GT)'], row['PT08.S1(CO)'], row['PT08.S2(NMHC)'], row['PT08.S3(NOx)'], row['PT08.S4(NO2)'], 
                              row['PT08.S5(O3)'], row['RH'], row['T'], row['AH']))
    
    if i % 100 == 0:  # Chèn mỗi 100 dòng
        session.execute(batch)
        batch = BatchStatement()  # Reset batch

if batch:
    session.execute(batch)

print("✅ Dữ liệu đã được nạp vào Cassandra thành công!")

✅ Dữ liệu đã được nạp vào Cassandra thành công!
